In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_2007_2015_bn.csv")

In [3]:
df_pl = df_pl.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_pl

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,PR,RQ,COC,ROL,CPI,EUPC
0,5.83,2.57,3.63,14.74,124.0,-0.05,29.28,25.418646,18.166685,6.818440,...,55.6,-0.01,868.25,535.0,50,0.77,0.28,0.41,42.0,3.674718e+07
1,2.72,1.91,3.69,13.97,124.0,0.01,28.82,24.756145,18.626276,4.369536,...,55.6,0.23,1382.84,-6200.0,50,0.84,0.45,0.55,46.0,5.751207e+07
2,3.18,2.06,3.74,13.28,125.0,0.07,29.96,20.757914,18.707747,2.545617,...,55.6,0.31,1862.41,-14768.0,50,0.98,0.44,0.65,50.0,1.351421e+08
3,3.94,2.23,3.65,13.05,124.0,-0.29,29.70,20.858669,19.149239,3.464009,...,55.6,0.22,1815.10,18440.0,55,1.02,0.51,0.71,53.0,1.800988e+08
4,3.57,2.33,3.65,12.91,124.0,0.05,30.39,22.345046,18.077861,5.198882,...,55.8,0.07,2778.30,-3920.0,60,0.96,0.56,0.79,55.0,2.473870e+08
5,1.53,2.62,3.81,12.58,124.0,-0.00,30.23,20.801952,18.008657,1.512024,...,56.2,0.13,4021.63,-6980.0,60,1.00,0.66,0.82,58.0,2.653231e+08
6,0.26,3.05,3.91,12.00,124.0,-0.06,28.68,19.128045,18.330801,0.745032,...,56.3,0.04,-269.29,-5791.0,60,1.05,0.65,0.84,60.0,2.753868e+08
7,3.85,3.55,4.02,11.49,124.0,-0.07,29.94,20.827279,18.238027,3.998318,...,58.2,0.12,3032.62,-13172.0,60,1.06,0.68,0.87,61.0,3.074712e+08
8,3.30,3.81,4.07,11.53,124.0,-0.07,30.99,20.801696,17.905031,4.501385,...,59.9,-0.05,3229.87,3157.0,60,0.95,0.69,0.80,62.0,1.699951e+08


In [6]:
discretised_pl = pd.DataFrame(index=df_pl.index)

for col in df_pl.columns:
    no_unique = df_pl[col].nunique()
    
    if no_unique <= 1:
        discretised_pl[col] = 0
    else:
        try:
            discretised_pl[col] = pd.qcut(
                df_pl[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_pl[col] = df_pl[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_pl)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    2    1    0     2     0    1     0     2     1      2  ...    0    0   
1    0    0    1     2     0    2     0     2     2      1  ...    0    2   
2    1    0    1     2     0    2     1     0     2      0  ...    0    2   
3    2    0    0     1     0    0     1     1     2      1  ...    0    2   
4    1    1    0     1     0    2     2     2     0      2  ...    0    1   
5    0    1    1     1     0    1     2     1     0      0  ...    0    1   
6    0    2    2     0     0    1     0     0     1      0  ...    1    0   
7    2    2    2     0     0    0     1     1     1      1  ...    1    1   
8    1    2    2     0     0    0     2     0     0      2  ...    1    0   

   EPFRG  GGAG  PR  RQ  COC  ROL  CPI  EUPC  
0      0     2   0   0    0    0    0     0  
1      0     1   0   0    0    0    0     0  
2      1     0   0   1    0    0    0     0  
3      1     2   1   2    1

In [7]:
sm = from_pandas(discretised_pl)

In [8]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_PL.html")

Graphs/01_fully_connected_PL.html


In [9]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_PL.html")

Graphs/01_thresholded_PL.html


In [10]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_PL_1.html")

In [11]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

     From     To    Weight
43     PR   GGAG  5.216984
28  EPFRG    FDI  3.989676
36     PR    FDI  2.406773
23    FEM    FDI  2.400519
50    COC  GDPCG  2.245212
38     PR   VABI  2.233035
6     QOA   EIAP  1.999415
0     QOR    FDI  1.996655
14   EBSG    PGP  1.879521
17   LBGD    EMP  1.766973
9   GDPCG   GCFP  1.759737
54    COC  EPFRG  1.607400
29  EPFRG   VABI  1.487992
35  EPFRG   GGAG  1.462958
37     PR   EIAP  1.374271
53    COC   LBGD  1.329114
48    COC   GCFP  1.229384
34  EPFRG    LFG  1.177299
18    EMP   GGAG  1.145400
1     QOR   EIAP  1.144939
22    LTU     RQ  1.108202
20    LTU   GCFP  1.055641
16   EBSG     RQ  1.018142
27    FEM    LFG  0.992747
40     PR  GDPCG  0.991494
10    EII  GDPCG  0.979842
42     PR    LFG  0.967564
12   EBSG    QOA  0.937505
30  EPFRG   GGFC  0.889000
26    FEM   EBSG  0.888988
2     QOR   LBGD  0.863835
32  EPFRG   EBSG  0.842673
19    LTU    FDI  0.825685
8    VABI    EMP  0.808924
7    VABI   GGFC -0.893385
15   EBSG  GDPCG -0.899134
4